# Harvest Now, Decrypt Later (Shor + PQC)

*Secure the Grid, step 2 · UTC Problem 4*

Most of this problem is analysis. The code is a small proof of concept: factor 15 with Shor's algorithm (Part 1), and list the quantum computers available on qBraid (Part 2). Parts 3 and 4, the exposure map and the migration plan, are in `WRITEUP.md`.

> **Kernel:** use **qBraid-SDK (v0.12.2)**. If *Python 3 (ipykernel)* is selected when the notebook opens, click the kernel name at the top right and switch. The first cell checks the kernel and stops with a message if it's wrong.

In [1]:
# Use the qBraid-SDK (v0.12.2) kernel. This stops with a clear message if another one is picked.
import importlib.util
missing = [m for m in ("qbraid", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Wrong kernel: this one has no " + " or ".join(missing) + ". Click the kernel name "
                      "at the top right of the notebook, pick qBraid-SDK (v0.12.2), then run this cell again.")

# Qiskit Aer is Qiskit's simulator. It runs right here in your Lab instance: instant, free and noise-free,
# with no job and no credits. The first run installs it (a few seconds), pinned to the version tested here.
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()

print("Setup OK: the qBraid kernel and Qiskit Aer are ready. You're good to go.")

Setup OK: the qBraid kernel and Qiskit Aer are ready. You're good to go.


### Part 1: Shor demo (factor 15)

Quantum period-finding for N = 15 with base a = 7 (a standard textbook case), followed by the classical step that turns the period into the two factors.

`c_amod15_a7` is hard-coded for N = 15: multiplying by 7 mod 15 happens to be a fixed pattern of swaps and X gates. It would not work for N = 21. A 2048-bit number needs a general modular-exponentiation circuit with billions of fault-tolerant gates. So the demo shows how period-finding works, not that Shor's algorithm runs at scale.

In [2]:
def c_amod15_a7(power):
    """Controlled multiplication by 7^power mod 15 (precompiled for the demo)."""
    U = QuantumCircuit(4)
    for _ in range(power):
        U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        for q in range(4):
            U.x(q)
    U = U.to_gate(); U.name = f"7^{power} mod 15"
    return U.control()

def qft_dagger(n):
    qc = QuantumCircuit(n)
    for q in range(n // 2):
        qc.swap(q, n - q - 1)
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi / 2 ** (j - m), m, j)
        qc.h(j)
    qc.name = "QFT†"
    return qc


In [3]:
n_count = 8                      # counting qubits
qc = QuantumCircuit(n_count + 4, n_count)
qc.h(range(n_count))
qc.x(n_count)                    # set work register to |1>
for i in range(n_count):
    qc.append(c_amod15_a7(2 ** i), [i] + list(range(n_count, n_count + 4)))
qc.append(qft_dagger(n_count), range(n_count))
qc.measure(range(n_count), range(n_count))

counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
measured = max(counts, key=counts.get)
phase = int(measured, 2) / (2 ** n_count)
print("Measured phase:", phase)


Measured phase: 0.5


In [4]:
from fractions import Fraction
from math import gcd

# Each run lands on one of four phases (0, 1/4, 1/2, 3/4) about equally often.
# Phases 0 and 1/2 give a wrong period, so try the results from most to least common
# and keep the first r that is even and passes the check 7^r mod 15 = 1.
ranked = sorted(counts, key=counts.get, reverse=True)
for measured in ranked:
    phase = int(measured, 2) / (2 ** n_count)
    r = Fraction(phase).limit_denominator(15).denominator
    print(f"measured {measured} -> phase {phase} -> candidate period r = {r}, 7^r mod 15 = {pow(7, r, 15)}")
    if r % 2 == 0 and pow(7, r, 15) == 1:
        break

# Use the period r to compute the factors of 15.
# For even r with 7^(r/2) != -1 (mod 15), gcd(7^(r/2) -/+ 1, 15) gives the two non-trivial factors.
assert r % 2 == 0 and pow(7, r // 2, 15) != 15 - 1, "this period can't give factors; rerun for another sample"
f1 = gcd(7 ** (r // 2) - 1, 15)
f2 = gcd(7 ** (r // 2) + 1, 15)
print(f"r = {r}  ->  15 = {f1} x {f2}")
assert f1 * f2 == 15 and 1 < f1 < 15


measured 10000000 -> phase 0.5 -> candidate period r = 2, 7^r mod 15 = 4
measured 11000000 -> phase 0.75 -> candidate period r = 4, 7^r mod 15 = 1
r = 4  ->  15 = 3 x 5


### Part 2: how far is real hardware?

The gate-based quantum computers on qBraid and their qubit counts, to compare with published estimates for breaking RSA-2048.

In [5]:
# List qBraid's gate-based quantum computers and their qubit counts
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()
analog = ("quera:qpu:aquila", "pasqal:qpu:fresnel")      # analog machines: they can't run Shor's circuits
for d in provider.get_devices():
    if ":qpu:" in d.id and not d.id.endswith(analog):
        print(f"{d.id:42s} {d.num_qubits} qubits")
# The same machine can appear under several routes (direct, aws, azure, openquantum): count it once.

# Results (run 13 Nov 2026; sources in WRITEUP.md, Step 2, Part 2):
#  - Largest machine in this list: 107 qubits (Rigetti Cepheus-1-108Q). qBraid's web catalog also lists IBM
#    machines with 156 qubits (Fez, Marrakesh, Kingston, Boston), which this SDK listing doesn't show.
#  - RSA-2048: under 1 million noisy physical qubits (about 1,400 logical), in under a week (Gidney 2025,
#    arXiv:2505.15917). The 2019 estimate was 20 million (Gidney & Ekerå). Alice & Bob (IACR ePrint 2026/2665)
#    estimate about 22,593 cat qubits, a different kind of qubit with its own noise assumptions.
#  - Gap: 1,000,000 / 156 ≈ 6,400x and 1,000,000 / 107 ≈ 9,300x, so about 4 orders of magnitude.
#  - A qubit count leaves out error rates, connectivity, coherence time, and the error-corrected logical
#    qubits needed to run for days.

aws:aqt:qpu:ibex-q1                        12 qubits
aws:iqm:qpu:emerald                        54 qubits
azure:ionq:qpu:forte-1                     36 qubits
aws:ionq:qpu:forte-1                       36 qubits
aws:ionq:qpu:forte-enterprise-1            36 qubits
ionq:ionq:qpu:forte-1                      36 qubits
aws:iqm:qpu:garnet                         20 qubits
azure:rigetti:qpu:cepheus-1-36q            36 qubits
azure:ionq:qpu:forte-enterprise-1          36 qubits
rigetti:rigetti:qpu:cepheus-1-108q         107 qubits
aws:rigetti:qpu:cepheus-1-108q             107 qubits
openquantum:aqt:qpu:ibex-q1                12 qubits
openquantum:ionq:qpu:forte-1               36 qubits
openquantum:ionq:qpu:forte-enterprise      36 qubits
openquantum:iqm:qpu:emerald                54 qubits
openquantum:iqm:qpu:garnet                 20 qubits
openquantum:rigetti:qpu:cepheus-1-108q     107 qubits


## Results

The answers are in `WRITEUP.md`, Step 2.
- **Part 1:** period r = 4, so 15 = 3 × 5.
- **Part 2:** the largest machine has 156 qubits (107 in this SDK list), against under 1 million physical qubits for RSA-2048: a gap of about 4 orders of magnitude.
- **Parts 3 and 4:** the exposure map and the migration plan, in the write-up.
- **Part 5:** `2b_grover_symmetric_keys.ipynb`.